# 00 — Launch & verify (run this first)

**Purpose of this notebook: prove the pipeline actually works, end to end, before anything else gets built on top of it.**

Everything in `src/qdot_edu/` has been ported, refactored, and statically reviewed — but **not yet executed**. That includes QArray's exact API shape and an assumption about un-swept gate voltages (see `docs/PORTING_NOTES.md`, item 6, and `model_params.py`'s verification caveat). This notebook is a deliberate smoke test, cheapest-thing-first:

1. Install
2. Import check (isolate `qarray`/`torch`/etc. failures from pipeline logic failures)
3. No-UI pipeline run (a handful of frames, `serial` mode — the simplest path)
4. Launch the Streamlit app through a tunnel

**If any step here fails, stop and fix it here.** Don't move on to the tutorial notebooks (`01`–`07`) against a pipeline that hasn't been proven to run. See `docs/lessons/README.md` for why.

## Step 0 — Get this repo into Colab

Clones the repo and remembers the path with `%store` so it survives runtime restarts (see the "Restore working directory" cell after Step 1 -- `%cd` alone only lasts for the current kernel session and gets silently wiped by a restart, which is what broke relative paths in Steps 3/5 on the first real run).

In [ ]:
!git clone https://github.com/k1151msarandega/SimQuantum-AMD-Developer-Hackathon.git
%cd SimQuantum-AMD-Developer-Hackathon/WISER26

# Sanity check: confirm we're in the right place before proceeding.
import os
assert os.path.exists("pyproject.toml"), (
    "pyproject.toml not found in the current directory -- Step 0 above "
    "didn't actually get you into the WISER26/ repo root. Fix that before continuing."
)
print("OK: pyproject.toml found, working directory looks right:", os.getcwd())

# Persist the resolved absolute path across runtime restarts (see note above).
project_root = os.getcwd()
%store project_root

## Step 1 — Install

**Revised after a real run surfaced a bug here:** installing torch separately first (via the CPU-only wheel index) collided with `qarray`'s exact `numpy==2.2.4` pin -- pip downgraded numpy for torch, uninstalled `scipy` as a side effect, and never fully reinstalled either. One combined `pip install -e .` call lets pip resolve the whole dependency graph at once instead of colliding with itself across two separate calls.

In [ ]:
# Single combined install -- see the markdown note above for why this
# isn't split into a separate torch + project install anymore.
# Colab's default runtime is CPU already (unless you explicitly selected
# a GPU runtime under Runtime > Change runtime type), so plain `torch` as
# listed in pyproject.toml resolves fine without forcing a specific index.
!pip install -q -e .

### Restart the runtime now

**Required.** Colab keeps already-imported C-extension modules (numpy/scipy) loaded in memory for the running session; a pip-level swap on disk doesn't take effect until the Python process restarts. Go to **Runtime → Restart session** now, then continue from the Step 2 cell below (you do NOT need to re-run Step 1).

**If Step 2 still fails after a restart with something like `ImportError: cannot import name '_center' from 'numpy._core.umath'`** -- that's not a caching issue, it's a genuinely corrupted numpy install on disk (leftover mixed-version compiled files from pip's earlier downgrade/upgrade churn; a plain reinstall or `--force-reinstall` can still reuse a bad cached wheel and not actually fix it). Run the repair cell below, which uninstalls first and disables the cache, then restart the runtime again.

In [ ]:
# Repair cell -- only run this if Step 2 still fails after a restart.
# Uninstalling first (not just --force-reinstall) and disabling the
# cache matters here: --force-reinstall alone can still reuse a
# corrupted/mismatched cached wheel and reproduce the same failure.
!pip uninstall -y -q numpy scipy
!pip install -q --no-cache-dir numpy==2.2.4 scipy
print("Done. Now: Runtime -> Restart session, then retry Step 2 (skip this cell and Step 1).")

### Restore working directory (run this every time, right after any runtime restart)

`%cd` from Step 0 only lasts for the kernel session that ran it -- a restart resets Colab's cwd back to `/content`. This cell reads back the `project_root` persisted in Step 0 via `%store` and `%cd`s into it again, so every relative path in Steps 2-5 works regardless of how many restarts happened in between. Cheap to run even if you're not sure it's needed.

In [ ]:
%store -r project_root
%cd $project_root

import os
assert os.path.exists("pyproject.toml"), (
    "Restored project_root doesn't look like the repo root -- if this is "
    "a brand new runtime that never ran Step 0, go run Step 0 first."
)
print("Working directory restored:", os.getcwd())

## Step 2 — Import check

Isolates "the environment/dependencies are broken" from "the pipeline logic is broken" -- if this cell fails, the problem is install/dependency related, not something in `src/qdot_edu/`.

In [ ]:
import qarray
print("qarray OK:", qarray.__file__)

import torch
print("torch OK, version:", torch.__version__)

import qdot_edu
from qdot_edu import model_params, pipeline
from qdot_edu.stream import generator, trajectory
from qdot_edu.twin import serial_estimator, batch_estimator, staleness
from qdot_edu.agent import triage, thresholds
from qdot_edu.perception import ensemble, ood
from qdot_edu.viz import potential_well, lab_theme
print("All qdot_edu submodules imported OK.")

## Step 3 — No-UI pipeline smoke test

Runs `serial` mode against `configs/trajectory_quick.yaml`, but truncated to a handful of frames first (not the full 300) so a failure here is fast and cheap to iterate on. This exercises the full real chain: QArray → ensemble → staleness logging -- no Streamlit, no tunnel, nothing to complicate debugging if something breaks.

In [ ]:
import yaml

# Build a tiny throwaway config (5 frames) from trajectory_quick.yaml so this
# first real run is as fast as possible to debug if it breaks.
with open("configs/trajectory_quick.yaml") as f:
    tiny_cfg = yaml.safe_load(f)
tiny_cfg["n_frames"] = 5
with open("configs/_smoke_test.yaml", "w") as f:
    yaml.dump(tiny_cfg, f)

print("Running pipeline.run('serial', ...) on a 5-frame smoke-test config...")
log = pipeline.run("serial", "configs/_smoke_test.yaml", device="cpu")
df = log.to_dataframe()
print(f"OK: {len(df)} frames processed.")
print(df)

If the cell above printed a small dataframe with `wall_clock_lag` values, the core pipeline works: QArray generated real stability-diagram patches, the ensemble model ran, and staleness was logged correctly.

**If it failed:** read the traceback carefully -- likely culprits, in rough order of likelihood, per `docs/PORTING_NOTES.md`:
- `qarray.DotArray(...)` or `.do2d_open(...)` raising -- the API assumed in `stream/generator.py` doesn't match the installed QArray version. Check the installed version's actual API (`help(qarray.DotArray)`).
- Gate name mismatches (`model_params.gate_names` produces `"P1"`, `"P2"`, ...) -- confirm QArray expects gate names in that exact string format.
- Import-only failures caught in Step 2 instead -- if Step 2 passed but this fails, the bug is in `stream/generator.py` or `pipeline.py`'s logic, not the environment.

## Step 4 — Try batched + triage too

Same idea, still no UI -- confirms the batching and triage-decision code paths work before layering the Streamlit app on top.

In [ ]:
print("Running pipeline.run_detailed('batched_triage', ...) on the same smoke-test config...")
log, tier_counts, max_q, events, tier_compute_s = pipeline.run_detailed(
    "batched_triage", "configs/_smoke_test.yaml", device="cpu"
)
print("tier_counts:", tier_counts)
print("max queue depth seen:", max_q)
print("tier_compute_s:", tier_compute_s)
print(log.to_dataframe())

## Step 5 — Launch the Streamlit app

Colab can't serve Streamlit directly, so this launches it in the background and exposes it through `localtunnel`. Only run this once Steps 1-4 above have passed clean.

**Revised after a real run:** a fixed `sleep 3` before starting the tunnel wasn't reliably long enough for Streamlit to finish starting -- the first real run showed a blank page, then several of the heavier widget bundles (the Vega-Lite staleness chart, the Plotly potential-well chart) failed with `TypeError: Failed to fetch dynamically imported module`, while lighter widgets (the sidebar selectboxes) eventually loaded after a couple of manual refreshes. That's the signature of the tunnel routing traffic before the server was fully up, not a code bug -- **the pipeline itself ran correctly** (real `tier_counts` output was visible). This cell now polls the port until Streamlit actually responds before starting the tunnel, instead of guessing a fixed delay.

In [ ]:
!pip install -q streamlit
get_ipython().system_raw("streamlit run app.py --server.port 8501 &> /content/streamlit_log.txt &")

# Poll until Streamlit actually answers on the port, instead of a fixed sleep --
# starting the tunnel before the server is ready is what caused the first
# real run's blank page + failed chart-widget loads. Up to ~30s.
import time
import urllib.request

for attempt in range(30):
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=1)
        print(f"Streamlit is up after ~{attempt + 1}s.")
        break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit didn't respond after 30s -- check /content/streamlit_log.txt "
          "before starting the tunnel (it will just tunnel to nothing).")

!npx --yes localtunnel --port 8501

Click the printed URL above to open the app. If `localtunnel` asks for a "Tunnel Password", it's the public IP `localtunnel` prints just before the link -- paste that in.

**If the page is blank or a widget shows `Failed to fetch dynamically imported module`:** try a hard refresh (Ctrl/Cmd+Shift+R, not a plain refresh -- a plain refresh can keep reusing a cached, broken reference to an old JS chunk) or open the link in an incognito window / different browser to rule out an extension interfering. If it persists after that, check the actual server-side error:
```python
!cat /content/streamlit_log.txt
```

## What to report back
If anything in Steps 1-5 failed, share: which step, the full traceback, and the installed `qarray` version (`!pip show qarray`). That's enough to fix the specific line rather than guessing.